# 00 — Design scorecard (claim D1)
Reproduces the diagnosability scorecard fixed before any candidate was computed (`docs/SCORECARD.md`) and
checks that the System III package reproduces the design toolkit (`design/`). numpy only, ~5 min.

In [ ]:
import json, sys, time
from dataclasses import replace
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
NB_DIR = Path.cwd().resolve()
assert NB_DIR.name == "notebooks" and NB_DIR.parent.name == "system_III", f"run from system_III/notebooks (cwd {NB_DIR})"
ROOT = NB_DIR.parent
sys.path.insert(0, str(ROOT / "src"))
import importlib, system_iii
from system_iii import physics as ph, simulator as sim, features as fe, data as da, scenarios as scn
from system_iii import sensors as sen, information as info, config as cfg, rules, metrics as wm
for _m in (ph, sim, fe, da, scn, sen, info, cfg, rules, wm):
    importlib.reload(_m)
R, FIGS, DATA = ROOT / "results", ROOT / "figures", ROOT / "data"
for _d in (R, FIGS, DATA):
    _d.mkdir(exist_ok=True)
P = ph.PLANT
T0 = time.time()

## 1. Steady states: uniqueness, safety, cooling margin, conversion

In [ ]:
import itertools
grid = np.linspace(0.4, 1.2, 17)
corners = [dict()] + [dict(Ti=P.Ti + a, Tci=P.Tci + b, Ci=P.Ci * (1 + c), Q=P.Q * (1 + q))
                      for a, b, c, q in itertools.product([-2, 2], [-2, 2], [-0.02, 0.02], [-0.01, 0.01])]
n = mult = sat = 0; Tmax = 0
for inl in corners:
    for a in grid:
        for b in grid:
            sols = ph.ss_closed(P, a, b, **inl); n += 1
            mult += len(sols) != 1; sat += sols[0]["sat"] == 1; Tmax = max(Tmax, max(s["T"] for s in sols))
s = ph.ss_unsat(P, 1.0, 1.0)
c1 = dict(n_points=n, n_multiple=mult, frac_saturated=sat / n, Tmax=Tmax, X_nominal=float(s["X"]),
          jacket_share=float(s["qj"] / s["qgen"]), coolant_rise=float(s["Tc"] - P.Tci),
          qc_fraction=float(s["Qc"] / P.Qcmax), beta_sat=ph.beta_saturation(P),
          beta_sat_worst=max(ph.beta_saturation(P, **c) for c in corners[1:]))
display(pd.Series(c1).round(4))

## 2. Design toolkit results (stored) and package verification

In [ ]:
for f in ["ss_checks.json", "ctrl_checks.json", "c6_fim.json", "c6_fim_extended.json", "c8_R3.json", "verify_package.json"]:
    p = ROOT / "design" / f
    print(f, "found" if p.exists() else "MISSING")
ver = json.loads((ROOT / "design" / "verify_package.json").read_text())
display(pd.DataFrame({k: v for k, v in ver.items() if k.startswith("crb")}).T)
display(pd.DataFrame(ver["proxy_rule"]).T.round(4))

In [ ]:
crb = info.bayes_crb()           # re-run with the package (~2 min): all calibration offsets marginalised
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in crb.items()})
da.save_json(R / "00_scorecard.json", dict(steady_state=c1, crb_all_offsets=crb))

## Interpretation — to be written after the run
Write toward the storyline (`docs/system_III_plan.md` §1) and update the claims ledger (§2).